In [1]:
import wrds
import pandas as pd
import numpy as np
import yaml
from datetime import datetime
from pathlib import Path

In [2]:
# Read username from config file
file_path = Path.cwd() / '..' / 'config' / 'credentials.yaml'
with open(file_path, 'r') as file:
    cred = yaml.safe_load(file)
username = cred['wrds']['username']

db = wrds.Connection(wrds_username=username)

Loading library list...
Done


In [3]:
print(db.list_tables(library='optionm'))

['borrate1996', 'borrate1997', 'borrate1998', 'borrate1999', 'borrate2000', 'borrate2001', 'borrate2002', 'borrate2003', 'borrate2004', 'borrate2005', 'borrate2006', 'borrate2007', 'borrate2008', 'borrate2009', 'borrate2010', 'borrate2011', 'borrate2012', 'borrate2013', 'borrate2014', 'borrate2015', 'borrate2016', 'borrate2017', 'borrate2018', 'borrate2019', 'borrate2020', 'borrate2021', 'borrate2022', 'borrate2023', 'borrate2024', 'borrate2025', 'country', 'currency', 'distrd', 'distribution', 'distribution_projection', 'distrprojd1996', 'distrprojd1997', 'distrprojd1998', 'distrprojd1999', 'distrprojd2000', 'distrprojd2001', 'distrprojd2002', 'distrprojd2003', 'distrprojd2004', 'distrprojd2005', 'distrprojd2006', 'distrprojd2007', 'distrprojd2008', 'distrprojd2009', 'distrprojd2010', 'distrprojd2011', 'distrprojd2012', 'distrprojd2013', 'distrprojd2014', 'distrprojd2015', 'distrprojd2016', 'distrprojd2017', 'distrprojd2018', 'distrprojd2019', 'distrprojd2020', 'distrprojd2021', 'dist

In [4]:
SECURITY_ID = 108105
SECURITY_NAME = "SPX" # S&P 500 Index

START_DATE = "2025-08-18"
END_DATE = "2025-08-20"

DAYS_TO_EXPIRY_MIN = 30
DAYS_TO_EXPIRY_MAX = 60

# Data Filtering Parameters

MAX_SPREAD_PCT = 0.1 # Maximum spread allowed as % of mid price
MIN_VOLUME = 1000 # Minimum volume required for the option to be considered
MIN_IMPLIED_VOLATILITY = 0.1 # Minimum implied volatility required for the option to be considered
MAX_IMPLIED_VOLATILITY = 0.5 # Maximum implied volatility required for the option to be considered
MIN_OPTION_PRICE = 0.01 # Minimum option price required for the option to be considered
MAX_MONEYNESS = 1 # Maximum moneyness required for the option to be considered (log-moneyness = ln(strike/spot))

# Output Parameters
OUTPUT_DIR = 'data/options_metrics_raw'
SAVE_CSV = True
SAVE_PLOT = True
SAVE_SUMMARY = True

In [5]:
option_query = f"""
SELECT
    date, exdate, strike_price/1000 as strike_price,
    impl_volatility, best_bid, best_offer,
    cp_flag, volume, open_interest,
    delta, gamma, theta, vega
FROM optionm.opprcd2025
WHERE secid = {SECURITY_ID}
    AND date BETWEEN '{START_DATE}' AND '{END_DATE}'
    AND (exdate - date) BETWEEN {DAYS_TO_EXPIRY_MIN} AND {DAYS_TO_EXPIRY_MAX}
    AND volume >= {MIN_VOLUME}
    AND impl_volatility BETWEEN {MIN_IMPLIED_VOLATILITY} AND {MAX_IMPLIED_VOLATILITY}
"""

options_data = db.raw_sql(option_query)
options_data.head()

,date,exdate,strike_price,impl_volatility,best_bid,best_offer,cp_flag,volume,open_interest,delta,gamma,theta,vega
0,2025-08-18,2025-09-19,6450.0,0.119781,101.2,101.7,C,1516.0,22263.0,0.547054,0.001759,-672.336,744.2731
1,2025-08-18,2025-09-19,6460.0,0.118275,94.8,95.4,C,1581.0,2725.0,0.52964,0.001789,-663.3961,747.44
2,2025-08-18,2025-09-19,6465.0,0.117538,91.7,92.3,C,4734.0,3687.0,0.520752,0.001803,-658.5584,748.5036
3,2025-08-18,2025-09-19,6470.0,0.116875,88.7,89.3,C,2686.0,5656.0,0.51175,0.001815,-653.7554,749.2008
4,2025-08-18,2025-09-19,6475.0,0.116154,85.7,86.3,C,1719.0,9580.0,0.50264,0.001827,-648.3953,749.5173


In [6]:
rate_query = f"""
SELECT *
FROM optionm.zerocd
WHERE date BETWEEN '{START_DATE}' AND '{END_DATE}'
    AND days BETWEEN {DAYS_TO_EXPIRY_MIN} AND {DAYS_TO_EXPIRY_MAX}
"""
rate_data = db.raw_sql(rate_query)
rate_data.head()




,date,days,rate
0,2025-08-18,30.0,4.737787
1,2025-08-18,60.0,4.675686
2,2025-08-19,30.0,4.737034
3,2025-08-19,60.0,4.674813
4,2025-08-20,30.0,4.722409


In [7]:
spot_query = f"""
SELECT date, close as spot_price
FROM optionm.secprd2025
WHERE secid = {SECURITY_ID}
    AND date BETWEEN '{START_DATE}' AND '{END_DATE}'
"""
spot_data = db.raw_sql(spot_query)
spot_data.head()

,date,spot_price
0,2025-08-18,6449.15
1,2025-08-19,6411.37
2,2025-08-20,6395.78


In [8]:
db.close()

In [9]:
df = options_data.merge(spot_data, on='date', how='left')
df = df.merge(rate_data, on='date', how='left')

In [10]:
df['mid_price'] = (df['best_bid'] + df['best_offer']) / 2
df['spread'] = (df['best_offer'] - df['best_bid'])
df['spread_pct'] = df['spread'] / df['mid_price']
df['moneyness'] = df['strike_price'] / df['spot_price']
df['date'] = pd.to_datetime(df['date'])
df['exdate'] = pd.to_datetime(df['exdate'])
df['tau (time to maturity)'] = (df['exdate'] - df['date']).dt.days / 365
df_processed = df[
    (df['spread_pct'] <= MAX_SPREAD_PCT) &
    (df['volume'] >= MIN_VOLUME) &
    (df['impl_volatility'] >= MIN_IMPLIED_VOLATILITY) &
    (df['impl_volatility'] <= MAX_IMPLIED_VOLATILITY) &
    (df['moneyness'] <= MAX_MONEYNESS)
]

df_processed.shape

(226, 21)